In [13]:
# Imports básicos
import xarray as xr
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import time
from datetime import datetime
import sys
import glob

warnings.filterwarnings('ignore')

print("=" * 70)
print("🎯 SSP SCENARIOS - ACCESS-CM2")
print("=" * 70)
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"xarray version: {xr.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

🎯 SSP SCENARIOS - ACCESS-CM2
Inicio: 2025-10-12 20:52:51
xarray version: 2025.1.2
pandas version: 2.3.3
numpy version: 2.3.3


In [14]:
# Definir rutas principales
base_dir = Path("/home/aninotna/magister/tesis/justh2_pipeline")
cmip6_data_dir = base_dir / "data/cmip6"

# Rutas de datos SSP por variable
ssp_data_paths = {
    'pr': cmip6_data_dir / "pr",
    'tasmin': cmip6_data_dir / "tmin",
    'tasmax': cmip6_data_dir / "tmax"
}

# Directorios de salida
out_dir = base_dir / "out"
bias_params_dir = out_dir / "bias_params/ACCESS-CM2"  # Parámetros ya entrenados
corrected_ssp_dir = out_dir / "corrected_ssp/ACCESS-CM2"
logs_dir = base_dir / "logs"

# Crear directorios de salida
for directory in [corrected_ssp_dir, logs_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("📁 CONFIGURACIÓN DE RUTAS:")
print(f"  • Base: {base_dir}")
print(f"  • CMIP6 data: {cmip6_data_dir}")
print(f"  • Bias params (entrenados): {bias_params_dir}")
print(f"  • Corrected SSP output: {corrected_ssp_dir}")
print(f"  • Logs: {logs_dir}")

# Verificar que los directorios existen
print(f"\n🔍 VERIFICACIÓN:")
print(f"  CMIP6 data existe: {'✅' if cmip6_data_dir.exists() else '❌'}")
print(f"  Bias params existe: {'✅' if bias_params_dir.exists() else '❌'}")

for var, path in ssp_data_paths.items():
    exists = '✅' if path.exists() else '❌'
    print(f"  {var} SSP data: {exists} ({path})")

📁 CONFIGURACIÓN DE RUTAS:
  • Base: /home/aninotna/magister/tesis/justh2_pipeline
  • CMIP6 data: /home/aninotna/magister/tesis/justh2_pipeline/data/cmip6
  • Bias params (entrenados): /home/aninotna/magister/tesis/justh2_pipeline/out/bias_params/ACCESS-CM2
  • Corrected SSP output: /home/aninotna/magister/tesis/justh2_pipeline/out/corrected_ssp/ACCESS-CM2
  • Logs: /home/aninotna/magister/tesis/justh2_pipeline/logs

🔍 VERIFICACIÓN:
  CMIP6 data existe: ✅
  Bias params existe: ✅
  pr SSP data: ✅ (/home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/pr)
  tasmin SSP data: ✅ (/home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/tmin)
  tasmax SSP data: ✅ (/home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/tmax)


In [15]:
# Configuración de escenarios y variables
ssp_scenarios = ['ssp245', 'ssp370', 'ssp585']
model_name = 'ACCESS-CM2'

# Configuración región Valle de Aconcagua
valle_aconcagua_bounds = {
    'lat_min': -33.27,
    'lat_max': -32.26,
    'lon_min': -71.89,
    'lon_max': -70.00
}

print("⚙️ CONFIGURACIÓN SSP:")
print(f"📊 Modelo: {model_name}")
print(f"🔮 Escenarios: {ssp_scenarios}")
print(f"📈 Variables: {list(variable_config.keys())}")

print(f"\n📍 Región Valle de Aconcagua:")
for key, value in valle_aconcagua_bounds.items():
    print(f"    {key}: {value}")


⚙️ CONFIGURACIÓN SSP:
📊 Modelo: ACCESS-CM2
🔮 Escenarios: ['ssp245', 'ssp370', 'ssp585']
📈 Variables: ['pr', 'tasmin', 'tasmax']

📍 Región Valle de Aconcagua:
    lat_min: -33.27
    lat_max: -32.26
    lon_min: -71.89
    lon_max: -70.0


In [18]:
# Explorar archivos SSP disponibles
print("=" * 70)
print("📂 EXPLORANDO ARCHIVOS SSP DISPONIBLES")
print("=" * 70)

available_files = {}

for var, var_path in ssp_data_paths.items():
    print(f"\n--- Variable: {var.upper()} ---")
    print(f"Directorio: {var_path}")
    
    if not var_path.exists():
        print(f"  ❌ Directorio no encontrado")
        continue
        
    available_files[var] = {}
    
    for scenario in ssp_scenarios:
        scenario_path = var_path / scenario
        print(f"\n  📁 Escenario: {scenario.upper()}")
        
        if not scenario_path.exists():
            print(f"    ❌ Directorio {scenario} no encontrado")
            continue
            
        # Buscar archivos ACCESS-CM2 (archivos .nc que contengan ACCESS-CM2)
        nc_files = list(scenario_path.glob("*ACCESS-CM2*.nc"))
        
        if nc_files:
            available_files[var][scenario] = sorted(nc_files)
            print(f"    ✅ Archivos encontrados: {len(nc_files)}")
            for file in nc_files:
                print(f"      📄 {file.name}")
        else:
            print(f"    ❌ No se encontraron archivos ACCESS-CM2")
            
            # Mostrar todos los archivos disponibles para debug
            all_files = list(scenario_path.glob("*.nc"))
            if all_files:
                print(f"    🔍 Archivos disponibles:")
                for file in all_files[:3]:  # Mostrar solo los primeros 3
                    print(f"      📄 {file.name}")
                if len(all_files) > 3:
                    print(f"      ... y {len(all_files)-3} más")



📂 EXPLORANDO ARCHIVOS SSP DISPONIBLES

--- Variable: PR ---
Directorio: /home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/pr

  📁 Escenario: SSP245
    ✅ Archivos encontrados: 2
      📄 pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc
      📄 pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20640706-21001231.nc

  📁 Escenario: SSP370
    ✅ Archivos encontrados: 2
      📄 pr_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20150101-20640705.nc
      📄 pr_day_ACCESS-CM2_ssp370_r1i1p1f1_gn_20640706-21001231.nc

  📁 Escenario: SSP585
    ✅ Archivos encontrados: 2
      📄 pr_day_ACCESS-CM2_ssp585_r1i1p1f1_gn_20150101-20640705.nc
      📄 pr_day_ACCESS-CM2_ssp585_r1i1p1f1_gn_20640706-21001231.nc

--- Variable: TASMIN ---
Directorio: /home/aninotna/magister/tesis/justh2_pipeline/data/cmip6/tmin

  📁 Escenario: SSP245
    ✅ Archivos encontrados: 2
      📄 tasmin_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc
      📄 tasmin_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20640706-21001231.nc

  📁 Escenario: SSP3

In [ ]:
# Resumen de archivos disponibles
print("=" * 70)
print("📊 RESUMEN DE ARCHIVOS SSP DISPONIBLES")
print("=" * 70)

total_files = 0
for var in available_files:
    var_total = sum(len(files) for files in available_files[var].values())
    total_files += var_total
    print(f"\n{var.upper()}: {var_total} archivos")
    for scenario in available_files[var]:
        print(f"  {scenario}: {len(available_files[var][scenario])} archivos")

print(f"\n{'='*70}")
print(f"📦 TOTAL: {total_files} archivos SSP encontrados")
print(f"✅ Cobertura: 3 variables × 3 escenarios = 9 combinaciones")
print(f"📅 Período típico: 2015-2100 (dividido en 2 archivos por escenario)")
print(f"{'='*70}")

## Inspección Detallada de Archivos SSP

Vamos a abrir un archivo de ejemplo de cada variable para revisar:
- Estructura dimensional (time, lat, lon)
- Rango temporal exacto
- Resolución espacial
- Unidades
- Metadatos relevantes
- Coordenadas (para verificar que corresponden a la región)

In [19]:
# Inspeccionar archivos de ejemplo (primer archivo de cada variable en SSP245)
print("=" * 70)
print("🔍 INSPECCIÓN DETALLADA - ARCHIVOS SSP245 (Primer Período)")
print("=" * 70)

sample_files_info = {}

for var in ['pr', 'tasmin', 'tasmax']:
    if var not in available_files or 'ssp245' not in available_files[var]:
        print(f"\n❌ {var.upper()}: No disponible")
        continue
    
    # Tomar el primer archivo de SSP245
    sample_file = available_files[var]['ssp245'][0]
    
    print(f"\n{'='*70}")
    print(f"📄 VARIABLE: {var.upper()}")
    print(f"{'='*70}")
    print(f"Archivo: {sample_file.name}")
    
    try:
        # Abrir archivo
        ds = xr.open_dataset(sample_file)
        
        # Información básica
        print(f"\n📊 ESTRUCTURA:")
        print(f"  Dimensiones: {dict(ds.dims)}")
        print(f"  Variables: {list(ds.data_vars.keys())}")
        print(f"  Coordenadas: {list(ds.coords.keys())}")
        
        # Rango temporal
        print(f"\n📅 RANGO TEMPORAL:")
        time_min = ds.time.min().values
        time_max = ds.time.max().values
        print(f"  Inicio: {pd.Timestamp(time_min).strftime('%Y-%m-%d')}")
        print(f"  Fin: {pd.Timestamp(time_max).strftime('%Y-%m-%d')}")
        print(f"  Total días: {len(ds.time)}")
        
        # Rango espacial
        print(f"\n📍 RANGO ESPACIAL:")
        print(f"  Lat: {float(ds.lat.min()):.3f} a {float(ds.lat.max()):.3f}")
        print(f"  Lon: {float(ds.lon.min()):.3f} a {float(ds.lon.max()):.3f}")
        print(f"  Puntos lat: {len(ds.lat)}")
        print(f"  Puntos lon: {len(ds.lon)}")
        
        # Resolución aproximada
        if len(ds.lat) > 1 and len(ds.lon) > 1:
            lat_res = abs(float(ds.lat[1]) - float(ds.lat[0]))
            lon_res = abs(float(ds.lon[1]) - float(ds.lon[0]))
            print(f"  Resolución: ~{lat_res:.3f}° lat × {lon_res:.3f}° lon")
        
        # Variable principal
        var_name = list(ds.data_vars.keys())[0]
        var_data = ds[var_name]
        
        print(f"\n📈 VARIABLE '{var_name}':")
        print(f"  Forma: {var_data.shape}")
        print(f"  Tipo: {var_data.dtype}")
        print(f"  Unidades: {var_data.attrs.get('units', 'No especificadas')}")
        print(f"  Nombre largo: {var_data.attrs.get('long_name', 'No disponible')}")
        
        # Rango de valores (muestra pequeña)
        sample_data = var_data.isel(time=0).load()
        print(f"\n🔢 VALORES (primer día):")
        print(f"  Mínimo: {float(sample_data.min()):.4f}")
        print(f"  Máximo: {float(sample_data.max()):.4f}")
        print(f"  Media: {float(sample_data.mean()):.4f}")
        print(f"  NaN: {sample_data.isnull().sum().values} ({float(sample_data.isnull().sum())/sample_data.size*100:.2f}%)")
        
        # Metadatos relevantes
        print(f"\n📝 METADATOS:")
        attrs_to_show = ['source', 'institution', 'experiment', 'variant_label', 'grid_label']
        for attr in attrs_to_show:
            if attr in ds.attrs:
                value = ds.attrs[attr]
                if len(str(value)) > 60:
                    value = str(value)[:60] + "..."
                print(f"  {attr}: {value}")
        
        # Verificar si cubre Valle de Aconcagua
        print(f"\n🌎 COBERTURA VALLE DE ACONCAGUA:")
        lat_in_bounds = (float(ds.lat.min()) <= valle_aconcagua_bounds['lat_min'] and 
                        float(ds.lat.max()) >= valle_aconcagua_bounds['lat_max'])
        lon_in_bounds = (float(ds.lon.min()) <= valle_aconcagua_bounds['lon_min'] and 
                        float(ds.lon.max()) >= valle_aconcagua_bounds['lon_max'])
        
        if lat_in_bounds and lon_in_bounds:
            print(f"  ✅ Cubre la región completa")
        else:
            print(f"  ⚠️ Cobertura parcial o fuera de región")
            print(f"     Lat Valle: {valle_aconcagua_bounds['lat_min']} a {valle_aconcagua_bounds['lat_max']}")
            print(f"     Lon Valle: {valle_aconcagua_bounds['lon_min']} a {valle_aconcagua_bounds['lon_max']}")
        
        # Tamaño del archivo
        print(f"\n💾 TAMAÑO:")
        file_size_mb = sample_file.stat().st_size / 1024**2
        data_memory_mb = var_data.nbytes / 1024**2
        print(f"  Archivo en disco: {file_size_mb:.1f} MB")
        print(f"  En memoria: {data_memory_mb:.1f} MB")
        
        # Guardar info
        sample_files_info[var] = {
            'file': sample_file,
            'dims': dict(ds.dims),
            'time_range': (time_min, time_max),
            'lat_range': (float(ds.lat.min()), float(ds.lat.max())),
            'lon_range': (float(ds.lon.min()), float(ds.lon.max())),
            'units': var_data.attrs.get('units', 'unknown'),
            'covers_region': lat_in_bounds and lon_in_bounds
        }
        
        # Cerrar dataset
        ds.close()
        
        print(f"\n✅ Inspección completada para {var.upper()}")
        
    except Exception as e:
        print(f"\n❌ Error inspeccionando {var.upper()}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
print(f"✅ INSPECCIÓN DETALLADA COMPLETADA")
print(f"{'='*70}")

🔍 INSPECCIÓN DETALLADA - ARCHIVOS SSP245 (Primer Período)

📄 VARIABLE: PR
Archivo: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc

📊 ESTRUCTURA:
  Dimensiones: {'time': 18084, 'bnds': 2, 'lat': 144, 'lon': 192}
  Variables: ['time_bnds', 'lat_bnds', 'lon_bnds', 'pr']
  Coordenadas: ['time', 'lat', 'lon']

📅 RANGO TEMPORAL:
  Inicio: 2015-01-01
  Fin: 2064-07-05
  Total días: 18084

📍 RANGO ESPACIAL:
  Lat: -89.375 a 89.375
  Lon: 0.938 a 359.062
  Puntos lat: 144
  Puntos lon: 192
  Resolución: ~1.250° lat × 1.875° lon

📈 VARIABLE 'time_bnds':
  Forma: (18084, 2)
  Tipo: datetime64[ns]
  Unidades: No especificadas
  Nombre largo: No disponible

🔢 VALORES (primer día):
  Mínimo: 1420070400000000000.0000
  Máximo: 1420156800000000000.0000
  Media: 1420113600000000000.0000
  NaN: 0 (0.00%)

📝 METADATOS:
  source: ACCESS-CM2 (2019): 
aerosol: UKCA-GLOMAP-mode
atmos: MetUM-H...
  institution: CSIRO (Commonwealth Scientific and Industrial Research Organ...
  experiment: update of 

In [20]:
# Verificar inconsistencias y compatibilidad con bias correction
print("=" * 70)
print("🔬 ANÁLISIS DE COMPATIBILIDAD Y POSIBLES INCONSISTENCIAS")
print("=" * 70)

issues_found = []

# 1. Verificar cobertura temporal entre archivos
print("\n1️⃣ VERIFICAR CONTINUIDAD TEMPORAL ENTRE ARCHIVOS")
for var in ['pr', 'tasmin', 'tasmax']:
    if var not in available_files or 'ssp245' not in available_files[var]:
        continue
    
    print(f"\n  {var.upper()}:")
    files = available_files[var]['ssp245']
    
    for i, file in enumerate(files):
        ds = xr.open_dataset(file)
        time_start = pd.Timestamp(ds.time.min().values)
        time_end = pd.Timestamp(ds.time.max().values)
        print(f"    Archivo {i+1}: {time_start.strftime('%Y-%m-%d')} a {time_end.strftime('%Y-%m-%d')}")
        
        # Verificar solapamiento/gap con archivo anterior
        if i > 0:
            prev_ds = xr.open_dataset(files[i-1])
            prev_end = pd.Timestamp(prev_ds.time.max().values)
            gap = (time_start - prev_end).days - 1
            
            if gap == 0:
                print(f"      ✅ Continuidad perfecta con archivo anterior")
            elif gap == 1:
                print(f"      ✅ Continuidad (diferencia de 1 día esperada)")
            elif gap > 1:
                print(f"      ⚠️ GAP de {gap} días con archivo anterior")
                issues_found.append(f"{var}: Gap de {gap} días entre archivos")
            elif gap < 0:
                print(f"      ⚠️ SOLAPAMIENTO de {abs(gap)} días con archivo anterior")
                issues_found.append(f"{var}: Solapamiento de {abs(gap)} días entre archivos")
            
            prev_ds.close()
        
        ds.close()

# 2. Verificar consistencia de unidades entre variables
print("\n\n2️⃣ VERIFICAR UNIDADES Y COMPATIBILIDAD CON BIAS CORRECTION")
expected_units = {
    'pr': ['kg m-2 s-1', 'mm/day', 'mm day-1'],
    'tasmin': ['K', 'degC', 'Celsius', '°C'],
    'tasmax': ['K', 'degC', 'Celsius', '°C']
}

for var in sample_files_info:
    print(f"\n  {var.upper()}:")
    actual_units = sample_files_info[var]['units']
    expected = expected_units.get(var, [])
    
    print(f"    Unidades actuales: {actual_units}")
    print(f"    Unidades esperadas: {', '.join(expected)}")
    
    if actual_units in expected:
        print(f"    ✅ Unidades compatibles")
    else:
        print(f"    ⚠️ Unidades requieren conversión")
        issues_found.append(f"{var}: Unidades '{actual_units}' requieren verificación")

# 3. Verificar resolución espacial vs CR2MET
print("\n\n3️⃣ VERIFICAR RESOLUCIÓN ESPACIAL (vs CR2MET ~0.05°)")
for var, info in sample_files_info.items():
    dims = info['dims']
    lat_range = info['lat_range']
    lon_range = info['lon_range']
    
    # Calcular resolución aproximada
    lat_span = abs(lat_range[1] - lat_range[0])
    lon_span = abs(lon_range[1] - lon_range[0])
    
    lat_res = lat_span / dims['lat'] if dims['lat'] > 0 else 0
    lon_res = lon_span / dims['lon'] if dims['lon'] > 0 else 0
    
    print(f"\n  {var.upper()}:")
    print(f"    Resolución: {lat_res:.4f}° lat × {lon_res:.4f}° lon")
    print(f"    Puntos: {dims['lat']} × {dims['lon']}")
    
    if lat_res > 0.2 or lon_res > 0.2:
        print(f"    ⚠️ Resolución gruesa - requerirá regridding extenso")
        issues_found.append(f"{var}: Resolución gruesa ({lat_res:.3f}° × {lon_res:.3f}°)")
    elif lat_res < 0.1 and lon_res < 0.1:
        print(f"    ✅ Resolución fina - regridding directo posible")
    else:
        print(f"    ✅ Resolución moderada - regridding estándar")

# 4. Verificar cobertura espacial
print("\n\n4️⃣ VERIFICAR COBERTURA ESPACIAL DEL VALLE DE ACONCAGUA")
all_cover = True
for var, info in sample_files_info.items():
    status = "✅" if info['covers_region'] else "❌"
    print(f"  {var.upper()}: {status} {'Cubre región' if info['covers_region'] else 'NO cubre región'}")
    if not info['covers_region']:
        all_cover = False
        issues_found.append(f"{var}: No cubre completamente Valle de Aconcagua")

if all_cover:
    print(f"\n  ✅ Todos los archivos cubren la región de interés")

# 5. Comparar dimensiones entre variables
print("\n\n5️⃣ VERIFICAR CONSISTENCIA DIMENSIONAL ENTRE VARIABLES")
dims_list = [info['dims'] for info in sample_files_info.values()]
vars_list = list(sample_files_info.keys())

if len(dims_list) > 1:
    first_dims = dims_list[0]
    consistent = all(d == first_dims for d in dims_list[1:])
    
    if consistent:
        print(f"  ✅ Todas las variables tienen dimensiones consistentes:")
        print(f"     {first_dims}")
    else:
        print(f"  ⚠️ INCONSISTENCIA DIMENSIONAL detectada:")
        for var, dims in zip(vars_list, dims_list):
            print(f"     {var}: {dims}")
        issues_found.append("Dimensiones inconsistentes entre variables")

# Resumen final
print("\n" + "="*70)
print("📋 RESUMEN DE REVISIÓN")
print("="*70)

if issues_found:
    print(f"\n⚠️ SE ENCONTRARON {len(issues_found)} POSIBLES PROBLEMAS:")
    for i, issue in enumerate(issues_found, 1):
        print(f"  {i}. {issue}")
else:
    print(f"\n✅ NO SE ENCONTRARON PROBLEMAS CRÍTICOS")

print(f"\n📌 RECOMENDACIONES:")
print(f"  1. Los archivos están divididos en 2 períodos - necesitarás concatenarlos")
print(f"  2. Verificar que las unidades sean compatibles con bias correction")
print(f"  3. Aplicar regridding a resolución CR2MET (0.05°) antes de bias correction")
print(f"  4. Verificar calendarios (pueden tener 365/360 días vs estándar)")

print("="*70)

🔬 ANÁLISIS DE COMPATIBILIDAD Y POSIBLES INCONSISTENCIAS

1️⃣ VERIFICAR CONTINUIDAD TEMPORAL ENTRE ARCHIVOS

  PR:
    Archivo 1: 2015-01-01 a 2064-07-05
    Archivo 2: 2064-07-06 a 2100-12-31
      ✅ Continuidad perfecta con archivo anterior

  TASMIN:
    Archivo 1: 2015-01-01 a 2064-07-05
    Archivo 2: 2064-07-06 a 2100-12-31
      ✅ Continuidad perfecta con archivo anterior

  TASMAX:
    Archivo 1: 2015-01-01 a 2064-07-05
    Archivo 2: 2064-07-06 a 2100-12-31
      ✅ Continuidad perfecta con archivo anterior


2️⃣ VERIFICAR UNIDADES Y COMPATIBILIDAD CON BIAS CORRECTION

  PR:
    Unidades actuales: unknown
    Unidades esperadas: kg m-2 s-1, mm/day, mm day-1
    ⚠️ Unidades requieren conversión

  TASMIN:
    Unidades actuales: unknown
    Unidades esperadas: K, degC, Celsius, °C
    ⚠️ Unidades requieren conversión

  TASMAX:
    Unidades actuales: unknown
    Unidades esperadas: K, degC, Celsius, °C
    ⚠️ Unidades requieren conversión


3️⃣ VERIFICAR RESOLUCIÓN ESPACIAL (vs CR

## 🔍 Investigación de Unidades "Unknown"

Vamos a abrir los archivos nuevamente y revisar en detalle TODOS los atributos para encontrar dónde están las unidades.

In [21]:
# Investigar en detalle los atributos de cada archivo para encontrar las unidades
print("=" * 70)
print("🔍 INVESTIGACIÓN DETALLADA DE METADATOS Y UNIDADES")
print("=" * 70)

for var in ['pr', 'tasmin', 'tasmax']:
    if var not in available_files or 'ssp245' not in available_files[var]:
        continue
    
    sample_file = available_files[var]['ssp245'][0]
    
    print(f"\n{'='*70}")
    print(f"📄 {var.upper()}: {sample_file.name}")
    print(f"{'='*70}")
    
    try:
        ds = xr.open_dataset(sample_file)
        
        # 1. Listar TODAS las variables del dataset
        print(f"\n📊 VARIABLES EN EL DATASET:")
        for var_name in ds.data_vars:
            print(f"  • {var_name}")
        
        # 2. Para cada variable de datos, mostrar TODOS sus atributos
        for var_name in ds.data_vars:
            var_data = ds[var_name]
            print(f"\n📈 VARIABLE '{var_name}':")
            print(f"  Tipo de dato: {var_data.dtype}")
            print(f"  Dimensiones: {var_data.dims}")
            print(f"  Forma: {var_data.shape}")
            
            print(f"\n  🏷️  TODOS LOS ATRIBUTOS:")
            if len(var_data.attrs) == 0:
                print(f"    ⚠️ No hay atributos en la variable")
            else:
                for attr_name, attr_value in var_data.attrs.items():
                    # Truncar valores muy largos
                    value_str = str(attr_value)
                    if len(value_str) > 100:
                        value_str = value_str[:100] + "..."
                    print(f"    {attr_name}: {value_str}")
            
            # Buscar específicamente 'units' en diferentes variaciones
            print(f"\n  🔎 BÚSQUEDA DE UNIDADES:")
            units_found = False
            for possible_key in ['units', 'Units', 'UNITS', 'unit', 'Unit']:
                if possible_key in var_data.attrs:
                    print(f"    ✅ Encontrado en '{possible_key}': {var_data.attrs[possible_key]}")
                    units_found = True
                    break
            
            if not units_found:
                print(f"    ❌ No se encontró atributo 'units' en ninguna variación")
        
        # 3. Verificar también los atributos globales del dataset
        print(f"\n🌐 ATRIBUTOS GLOBALES DEL DATASET:")
        print(f"  Total atributos: {len(ds.attrs)}")
        
        # Mostrar algunos atributos relevantes
        relevant_global_attrs = ['source', 'institution', 'experiment', 'frequency', 
                                 'table_id', 'variable_id', 'realm']
        
        for attr in relevant_global_attrs:
            if attr in ds.attrs:
                value = ds.attrs[attr]
                if len(str(value)) > 80:
                    value = str(value)[:80] + "..."
                print(f"  {attr}: {value}")
        
        # 4. Verificar coordenadas también
        print(f"\n📍 COORDENADAS:")
        for coord_name in ds.coords:
            coord = ds[coord_name]
            if 'units' in coord.attrs:
                print(f"  {coord_name}: units = {coord.attrs['units']}")
        
        ds.close()
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*70}")
print(f"✅ INVESTIGACIÓN COMPLETADA")
print(f"{'='*70}")

🔍 INVESTIGACIÓN DETALLADA DE METADATOS Y UNIDADES

📄 PR: pr_day_ACCESS-CM2_ssp245_r1i1p1f1_gn_20150101-20640705.nc

📊 VARIABLES EN EL DATASET:
  • time_bnds
  • lat_bnds
  • lon_bnds
  • pr

📈 VARIABLE 'time_bnds':
  Tipo de dato: datetime64[ns]
  Dimensiones: ('time', 'bnds')
  Forma: (18084, 2)

  🏷️  TODOS LOS ATRIBUTOS:
    ⚠️ No hay atributos en la variable

  🔎 BÚSQUEDA DE UNIDADES:
    ❌ No se encontró atributo 'units' en ninguna variación

📈 VARIABLE 'lat_bnds':
  Tipo de dato: float64
  Dimensiones: ('time', 'lat', 'bnds')
  Forma: (18084, 144, 2)

  🏷️  TODOS LOS ATRIBUTOS:
    ⚠️ No hay atributos en la variable

  🔎 BÚSQUEDA DE UNIDADES:
    ❌ No se encontró atributo 'units' en ninguna variación

📈 VARIABLE 'lon_bnds':
  Tipo de dato: float64
  Dimensiones: ('time', 'lon', 'bnds')
  Forma: (18084, 192, 2)

  🏷️  TODOS LOS ATRIBUTOS:
    ⚠️ No hay atributos en la variable

  🔎 BÚSQUEDA DE UNIDADES:
    ❌ No se encontró atributo 'units' en ninguna variación

📈 VARIABLE 'pr':
 

In [22]:
# Usar la representación completa de xarray para ver toda la estructura
print("=" * 70)
print("📋 ESTRUCTURA COMPLETA DE LOS ARCHIVOS (estilo ncdump)")
print("=" * 70)

for var in ['pr', 'tasmin', 'tasmax']:
    if var not in available_files or 'ssp245' not in available_files[var]:
        continue
    
    sample_file = available_files[var]['ssp245'][0]
    
    print(f"\n{'='*70}")
    print(f"📄 {var.upper()}")
    print(f"{'='*70}")
    
    try:
        ds = xr.open_dataset(sample_file)
        
        # Mostrar la representación completa del dataset
        print(ds)
        
        print(f"\n" + "-"*70)
        
        ds.close()
        
    except Exception as e:
        print(f"❌ Error: {e}")

print(f"\n{'='*70}")

📋 ESTRUCTURA COMPLETA DE LOS ARCHIVOS (estilo ncdump)

📄 PR
<xarray.Dataset> Size: 2GB
Dimensions:    (time: 18084, bnds: 2, lat: 144, lon: 192)
Coordinates:
  * time       (time) datetime64[ns] 145kB 2015-01-01T12:00:00 ... 2064-07-05...
  * lat        (lat) float64 1kB -89.38 -88.12 -86.88 ... 86.88 88.12 89.38
  * lon        (lon) float64 2kB 0.9375 2.812 4.688 6.562 ... 355.3 357.2 359.1
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) datetime64[ns] 289kB ...
    lat_bnds   (time, lat, bnds) float64 42MB ...
    lon_bnds   (time, lon, bnds) float64 56MB ...
    pr         (time, lat, lon) float32 2GB ...
Attributes: (12/47)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            ScenarioMIP
    branch_method:          standard
    branch_time_in_child:   60265.0
    branch_time_in_parent:  60265.0
    creation_date:          2019-11-08T10:43:20Z
    ...                     ...
    variable_id:            pr
    variant_label:        

## 📊 Análisis de los Resultados

### ✅ **BUENAS NOTICIAS: Las unidades SÍ existen**

El problema era que en la celda de inspección inicial estábamos tomando la **primera variable** de `ds.data_vars.keys()`, que resultó ser `time_bnds` (una variable auxiliar sin atributos), en lugar de la variable principal de datos.

### 🔍 **Lo que encontramos:**

#### **PR (Precipitación)**:
- ✅ **Unidades**: `kg m-2 s-1` (kilogramos por metro cuadrado por segundo)
- ✅ **Nombre estándar**: `precipitation_flux`
- ✅ **Forma**: (18084, 144, 192) = 18084 días × 144 latitudes × 192 longitudes
- 📝 **Nota**: Estas unidades son **compatibles** con bias correction, pero necesitaremos convertirlas a `mm/day` para que sean comparables con CR2MET

#### **TASMIN (Temperatura Mínima)**:
- ✅ **Unidades**: `K` (Kelvin)
- ✅ **Nombre estándar**: `air_temperature`
- ✅ **Descripción**: Daily Minimum Near-Surface Air Temperature
- 📝 **Nota**: Necesitaremos convertir de Kelvin a Celsius (restar 273.15)

#### **TASMAX (Temperatura Máxima)**:
- ✅ **Unidades**: `K` (Kelvin)
- ✅ **Nombre estándar**: `air_temperature`
- ✅ **Descripción**: Daily Maximum Near-Surface Air Temperature
- 📝 **Nota**: Necesitaremos convertir de Kelvin a Celsius (restar 273.15)

### 📐 **Estructura de los archivos:**

Cada archivo contiene:
1. **Variable principal** (pr, tasmin, tasmax) - con los datos climáticos
2. **Variables auxiliares** (time_bnds, lat_bnds, lon_bnds) - definen límites de celdas
3. **Coordenadas** (time, lat, lon, height) - con sus propias unidades

### 🎯 **Conclusión: La data está BUENA**

Los archivos son datos CMIP6 oficiales correctamente formateados:
- ✅ Metadatos completos (CF-1.7 CMIP-6.2)
- ✅ Unidades estándar (kg m-2 s-1 para pr, K para temperaturas)
- ✅ Nombres estándar siguiendo convenciones CF
- ✅ Continuidad temporal perfecta entre archivos
- ✅ Estructura dimensional consistente

### ⚙️ **Próximos pasos necesarios:**

1. **Conversión de unidades**:
   - pr: `kg m-2 s-1` → `mm/day` (multiplicar por 86400)
   - tasmin/tasmax: `K` → `°C` (restar 273.15)

2. **Regridding**: 
   - Resolución actual: ~1.24° × 1.87° (144×192 puntos globales)
   - Resolución objetivo: ~0.05° (CR2MET)
   - Necesitaremos interpolación conservativa

3. **Recorte espacial**:
   - Los archivos son globales
   - Necesitamos extraer solo Valle de Aconcagua

4. **Concatenación temporal**:
   - Unir los 2 archivos de cada escenario (2015-2064 + 2064-2100)

In [23]:
# Verificar las unidades CORRECTAS accediendo a la variable principal
print("=" * 70)
print("✅ VERIFICACIÓN CORREGIDA DE UNIDADES")
print("=" * 70)

corrected_units = {}

for var in ['pr', 'tasmin', 'tasmax']:
    if var not in available_files or 'ssp245' not in available_files[var]:
        continue
    
    sample_file = available_files[var]['ssp245'][0]
    
    try:
        ds = xr.open_dataset(sample_file)
        
        # Acceder DIRECTAMENTE a la variable principal (pr, tasmin, tasmax)
        var_data = ds[var]
        
        # Obtener unidades
        units = var_data.attrs.get('units', 'unknown')
        standard_name = var_data.attrs.get('standard_name', 'unknown')
        long_name = var_data.attrs.get('long_name', 'unknown')
        
        corrected_units[var] = units
        
        print(f"\n{var.upper()}:")
        print(f"  ✅ Unidades: {units}")
        print(f"  📝 Nombre estándar: {standard_name}")
        print(f"  📄 Nombre largo: {long_name}")
        
        # Verificar compatibilidad
        if var == 'pr':
            if units == 'kg m-2 s-1':
                print(f"  🔧 Conversión necesaria: × 86400 → mm/day")
            elif units in ['mm/day', 'mm day-1']:
                print(f"  ✅ Ya está en mm/day")
        else:  # tasmin, tasmax
            if units == 'K':
                print(f"  🔧 Conversión necesaria: - 273.15 → °C")
            elif units in ['degC', 'Celsius', '°C']:
                print(f"  ✅ Ya está en °C")
        
        ds.close()
        
    except Exception as e:
        print(f"\n❌ Error en {var}: {e}")

print(f"\n{'='*70}")
print(f"📊 RESUMEN DE UNIDADES ENCONTRADAS:")
print(f"{'='*70}")
for var, units in corrected_units.items():
    print(f"  {var}: {units}")

print(f"\n✅ Todas las variables tienen unidades correctamente definidas")
print(f"🔧 Necesitaremos hacer conversiones de unidades antes del bias correction")
print(f"{'='*70}")

✅ VERIFICACIÓN CORREGIDA DE UNIDADES

PR:
  ✅ Unidades: kg m-2 s-1
  📝 Nombre estándar: precipitation_flux
  📄 Nombre largo: Precipitation
  🔧 Conversión necesaria: × 86400 → mm/day

TASMIN:
  ✅ Unidades: K
  📝 Nombre estándar: air_temperature
  📄 Nombre largo: Daily Minimum Near-Surface Air Temperature
  🔧 Conversión necesaria: - 273.15 → °C

TASMAX:
  ✅ Unidades: K
  📝 Nombre estándar: air_temperature
  📄 Nombre largo: Daily Maximum Near-Surface Air Temperature
  🔧 Conversión necesaria: - 273.15 → °C

📊 RESUMEN DE UNIDADES ENCONTRADAS:
  pr: kg m-2 s-1
  tasmin: K
  tasmax: K

✅ Todas las variables tienen unidades correctamente definidas
🔧 Necesitaremos hacer conversiones de unidades antes del bias correction


## 🚀 Plan de Procesamiento Completo SSP

### 📋 **Pipeline de Procesamiento**

Ahora que verificamos que los datos están correctos, el flujo completo será:

```
DATOS SSP RAW (CMIP6)
    ↓
1. CARGAR Y CONCATENAR
   - Unir archivos por escenario (2015-2064 + 2064-2100)
   - Variables: pr, tasmin, tasmax
   ↓
2. CONVERSIÓN DE UNIDADES
   - pr: kg m-2 s-1 → mm/day (× 86400)
   - tasmin/tasmax: K → °C (- 273.15)
   ↓
3. REGRIDDING A CR2MET
   - Resolución actual: ~1.24° × 1.87°
   - Resolución objetivo: 0.05° (rejilla CR2MET)
   - Método: Interpolación conservativa (xESMF o rioxarray)
   ↓
4. RECORTE ESPACIAL
   - Región: Valle de Aconcagua
   - Bounds: lat (-33.27, -32.26), lon (-71.89, -70.00)
   ↓
5. BIAS CORRECTION
   - Cargar parámetros entrenados (del historical)
   - Re-entrenar modelos DQM/EQM con mismos parámetros
   - Aplicar corrección a datos SSP
   ↓
6. EXPORTAR DATOS CORREGIDOS
   - Formato: NetCDF (.nc)
   - Ubicación: out/corrected_ssp/ACCESS-CM2/{var}/{scenario}/
   - Nombres: {var}_ACCESS-CM2_{scenario}_bias_corrected_2015-2100.nc
```

### 📦 **Estructura de Exportación**

Los datos corregidos se guardarán en:

```
out/corrected_ssp/ACCESS-CM2/
├── pr/
│   ├── ssp245/
│   │   └── pr_ACCESS-CM2_ssp245_bias_corrected_2015-2100.nc
│   ├── ssp370/
│   │   └── pr_ACCESS-CM2_ssp370_bias_corrected_2015-2100.nc
│   └── ssp585/
│       └── pr_ACCESS-CM2_ssp585_bias_corrected_2015-2100.nc
├── tasmin/
│   ├── ssp245/
│   │   └── tasmin_ACCESS-CM2_ssp245_bias_corrected_2015-2100.nc
│   ├── ssp370/
│   └── ssp585/
└── tasmax/
    ├── ssp245/
    ├── ssp370/
    └── ssp585/
```

### 🎯 **Características de los Archivos Exportados**

Cada archivo NetCDF incluirá:

1. **Datos corregidos**: Variable climática con bias correction aplicado
2. **Dimensiones**: (time, lat, lon) en rejilla CR2MET
3. **Metadatos completos**:
   - Unidades: mm/day (pr) o °C (tas*)
   - Atributos CF-compliant
   - Historia de procesamiento
   - Información de bias correction
4. **Coordenadas**: lat, lon, time con atributos completos
5. **Compresión**: zlib=True, complevel=4 para reducir tamaño

### 📊 **Tamaño Estimado de Archivos**

- **Período**: 2015-2100 = 86 años ≈ 31,390 días
- **Región**: Valle de Aconcagua ≈ 20×35 puntos en rejilla 0.05°
- **Tamaño por variable/escenario**: ~50-100 MB (comprimido)
- **Total 9 archivos** (3 vars × 3 escenarios): ~500-900 MB

### 🔧 **Uso Posterior**

Los datos exportados servirán para:

1. **Análisis de indicadores climáticos** (extremos, sequías, etc.)
2. **Input para modelo Calliope** (energía H₂)
3. **Comparación entre escenarios** SSP245 vs SSP370 vs SSP585
4. **Análisis de tendencias temporales** 2015-2100
5. **Visualizaciones** y mapas

### ⚙️ **Próximo Notebook**

Crearemos: `01_ssp_preprocessing_and_bias_correction.ipynb` que implementará todo el pipeline completo.

In [24]:
# Resumen final de la revisión
print("=" * 70)
print("🎉 RESUMEN FINAL - REVISIÓN SSP COMPLETADA")
print("=" * 70)

print("\n✅ DATOS DISPONIBLES:")
print(f"  • Modelo: ACCESS-CM2")
print(f"  • Variables: pr, tasmin, tasmax")
print(f"  • Escenarios: SSP245, SSP370, SSP585")
print(f"  • Período: 2015-2100 (86 años)")
print(f"  • Archivos totales: {total_files}")

print("\n📊 CALIDAD DE DATOS:")
print(f"  ✅ Formato: NetCDF CMIP6 oficial")
print(f"  ✅ Metadatos: Completos y CF-compliant")
print(f"  ✅ Unidades: Correctamente definidas")
print(f"  ✅ Continuidad temporal: Perfecta")
print(f"  ✅ Dimensiones: Consistentes entre variables")

print("\n🔧 PROCESAMIENTO NECESARIO:")
print(f"  1️⃣  Concatenar archivos temporales (2 por escenario)")
print(f"  2️⃣  Convertir unidades (pr: ×86400, tas: -273.15)")
print(f"  3️⃣  Regridding a 0.05° (CR2MET)")
print(f"  4️⃣  Recortar a Valle de Aconcagua")
print(f"  5️⃣  Aplicar bias correction (DQM/EQM)")
print(f"  6️⃣  Exportar NetCDF corregidos")

print("\n📁 RUTAS DE EXPORTACIÓN:")
print(f"  Base: {corrected_ssp_dir}")
print(f"  Estructura: {{var}}/{{scenario}}/{{var}}_ACCESS-CM2_{{scenario}}_bias_corrected_2015-2100.nc")

print("\n💾 TAMAÑO ESTIMADO:")
print(f"  Por archivo: ~50-100 MB (comprimido)")
print(f"  Total (9 archivos): ~500-900 MB")

print("\n🚀 PRÓXIMO PASO:")
print(f"  Crear notebook: 01_ssp_preprocessing_and_bias_correction.ipynb")
print(f"  Implementar pipeline completo de procesamiento")

print("\n" + "=" * 70)
print("✅ Revisión completada exitosamente")
print("📝 Los datos SSP están listos para procesamiento")
print("=" * 70)

🎉 RESUMEN FINAL - REVISIÓN SSP COMPLETADA

✅ DATOS DISPONIBLES:
  • Modelo: ACCESS-CM2
  • Variables: pr, tasmin, tasmax
  • Escenarios: SSP245, SSP370, SSP585
  • Período: 2015-2100 (86 años)
  • Archivos totales: 0

📊 CALIDAD DE DATOS:
  ✅ Formato: NetCDF CMIP6 oficial
  ✅ Metadatos: Completos y CF-compliant
  ✅ Unidades: Correctamente definidas
  ✅ Continuidad temporal: Perfecta
  ✅ Dimensiones: Consistentes entre variables

🔧 PROCESAMIENTO NECESARIO:
  1️⃣  Concatenar archivos temporales (2 por escenario)
  2️⃣  Convertir unidades (pr: ×86400, tas: -273.15)
  3️⃣  Regridding a 0.05° (CR2MET)
  4️⃣  Recortar a Valle de Aconcagua
  5️⃣  Aplicar bias correction (DQM/EQM)
  6️⃣  Exportar NetCDF corregidos

📁 RUTAS DE EXPORTACIÓN:
  Base: /home/aninotna/magister/tesis/justh2_pipeline/out/corrected_ssp/ACCESS-CM2
  Estructura: {var}/{scenario}/{var}_ACCESS-CM2_{scenario}_bias_corrected_2015-2100.nc

💾 TAMAÑO ESTIMADO:
  Por archivo: ~50-100 MB (comprimido)
  Total (9 archivos): ~500-900 